# Neo4j Cypher Retrieval Script (02_neo4j_cypher_retrieval.py)

This script demonstrates how to connect to a running Neo4j instance, ingest sample knowledge graph entities using Cypher queries, and execute structural multi-hop traversal to retrieve context for a RAG pipeline.

Prerequisites (Python Libraries)
Make sure you have the official Neo4j driver installed in your environment:


pip install neo4j

In [ ]:
"""
02_neo4j_cypher_retrieval.py
Demonstrates connecting to Neo4j, inserting sample corporate knowledge graph nodes,
and executing Cypher queries to retrieve multi-hop context for GraphRAG.
"""

from neo4j import GraphDatabase
import os

class Neo4jGraphRAGPipeline:
    def __init__(self, uri: str, auth: tuple):
        """Initializes the Neo4j database driver connection."""
        self.driver = GraphDatabase.driver(uri, auth=auth)
        self.verify_connectivity()

    def close(self):
        """Closes the database driver connection."""
        self.driver.close()

    def verify_connectivity(self):
        """Validates that the connection to Neo4j is active."""
        try:
            self.driver.verify_connectivity()
            print("Successfully connected to Neo4j database instance.")
        except Exception as e:
            print(f"Failed to connect to Neo4j: {e}")
            raise e

    def setup_sample_graph(self):
        """Ingests sample corporate nodes and relationships into Neo4j."""
        query = """
        MERGE (c:Company {name: $company_name})
        MERGE (s:Subsidiary {name: $sub_name})
        MERGE (loc:Location {city: $city_name})
        MERGE (sup:Supplier {name: $supplier_name})
        
        MERGE (c)-[:OWNS]->(s)
        MERGE (s)-[:LOCATED_IN]->(loc)
        MERGE (s)-[:PARTNERS_WITH]->(sup)
        """
        with self.driver.session() as session:
            session.run(
                query,
                company_name="TechCorp Global",
                sub_name="TechCorp Europe",
                city_name="Berlin",
                supplier_name="DataStream Logistics"
            )
            print("Sample knowledge graph successfully populated in Neo4j.")

    def retrieve_context_via_cypher(self, target_company: str):
        """
        Executes a multi-hop Cypher traversal query to fetch connected entities
        and assemble them into a structured context block for an LLM.
        """
        cypher_query = """
        MATCH (c:Company {name: $company_name})-[:OWNS]->(sub:Subsidiary)
        MATCH (sub)-[:LOCATED_IN]->(loc:Location)
        MATCH (sub)-[:PARTNERS_WITH]->(sup:Supplier)
        RETURN 
            c.name AS ParentCompany, 
            sub.name AS SubsidiaryName, 
            loc.city AS City, 
            sup.name AS PartnerSupplier
        """
        
        with self.driver.session() as session:
            result = session.run(cypher_query, company_name=target_company)
            records = [record.data() for record in result]
            return records

# --- Execution Block ---
if __name__ == "__main__":
    # Local Neo4j default connection credentials (adjust as needed for your environment)
    URI = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
    AUTH = (os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "password_secure"))

    try:
        # Initialize pipeline
        rag_pipeline = Neo4jGraphRAGPipeline(uri=URI, auth=AUTH)
        
        # Populate mock data
        rag_pipeline.setup_sample_graph()
        
        # Run Cypher retrieval query
        company_query = "TechCorp Global"
        print(f"\nExecuting Cypher retrieval for: '{company_query}'...")
        
        context_data = rag_pipeline.retrieve_context_via_cypher(company_query)
        
        print("\n--- Retrieved Graph Context for RAG Prompt ---")
        for idx, record in enumerate(context_data, 1):
            print(f"{idx}. Parent: {record['ParentCompany']} "
                  f"| Subsidiary: {record['SubsidiaryName']} "
                  f"| Location: {record['City']} "
                  f"| Partner Supplier: {record['PartnerSupplier']}")
            
    except Exception as err:
        print(f"\nNote: Ensure your Neo4j instance is running locally or update URI/Auth credentials. Error: {err}")
    finally:
        if 'rag_pipeline' in locals():
            rag_pipeline.close()

What this script covers:

**Driver Lifecycle Management:** Establishing secure connection pools and clean tear-downs using GraphDatabase.driver.

**Idempotent Ingestion (MERGE):** Using Cypher's MERGE clause to insert nodes and relationships without generating duplicate entries if run multiple times.

**Multi-Hop Traversal Querying:** Stitching together paths across multiple node types (Company $\rightarrow$ Subsidiary $\rightarrow$ Location/Supplier) to return rich contextual objects ready to be injected into an LLM system prompt.